In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_mlir import fx

def export_model(model, data, filename="result.txt"):
    module = fx.export_and_import(
        model,
        random_data,
        output_type="linalg-on-tensors",
    )
    with open(filename, "w") as f:
        print(module.operation.get_asm())
        f.write(module.operation.get_asm())
        
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # in, out, kernel_size
        self.lienar = nn.Linear(10, 10)
    
    # x represents our data
    def forward(self, x):
        # Apply softmax to x
        output = self.lienar(x)
        output = output + x
        return output

# Equates to one random 28x28 image
random_data = torch.rand((20, 10))
network = Net()
export_model(network, random_data, filename="reuslt.txt")

#map = affine_map<(d0, d1) -> (d0, d1)>
#map1 = affine_map<(d0, d1) -> (d1)>
module {
  func.func @main(%arg0: tensor<20x10xf32>) -> tensor<20x10xf32> {
    %cst = arith.constant 0.000000e+00 : f32
    %cst_0 = arith.constant dense_resource<torch_tensor_10_10_torch.float32> : tensor<10x10xf32>
    %cst_1 = arith.constant dense_resource<torch_tensor_10_torch.float32> : tensor<10xf32>
    %0 = tensor.empty() : tensor<10x10xf32>
    %transposed = linalg.transpose ins(%cst_0 : tensor<10x10xf32>) outs(%0 : tensor<10x10xf32>) permutation = [1, 0] 
    %1 = tensor.empty() : tensor<20x10xf32>
    %2 = linalg.fill ins(%cst : f32) outs(%1 : tensor<20x10xf32>) -> tensor<20x10xf32>
    %3 = linalg.matmul ins(%arg0, %transposed : tensor<20x10xf32>, tensor<10x10xf32>) outs(%2 : tensor<20x10xf32>) -> tensor<20x10xf32>
    %4 = linalg.generic {indexing_maps = [#map, #map1, #map], iterator_types = ["parallel", "parallel"]} ins(%3, %cst_1 : tensor<20x10xf32>, tensor<10xf32>) outs(%1 : tensor<20x10xf32>)

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/usr/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


# torch-mlir

Uisng the output from above, we can convert